Lakehouse & Apache Iceberg / Delta Lake 面试知识点



## 1. Lakehouse Architecture 概述

 

- A Lakehouse combines the best of data warehouses and data lakes. 
- It stores data in open file formats like Parquet on cheap object storage such as S3, but adds a **transactional metadata layer** on top — that's where table formats like Iceberg and Delta Lake come in. 
- This gives you **ACID** transactions, **schema enforcement**, and **time travel**, while still allowing **direct access from ML frameworks and BI tools**. 
- The key insight is: you get **warehouse-like reliability** without duplicating data into a separate warehouse.

 

Lakehouse 架构的核心思想是**在 data lake 的廉价存储之上，加一层 transactional metadata layer**，从而获得 data warehouse 的可靠性。

传统架构的问题：
- **Data Lake**：存储便宜（S3/ADLS），但没有 ACID、没有 schema enforcement，数据质量难以保证。
- **Data Warehouse**：有 ACID 和 schema，但存储昂贵，且数据需要从 lake 复制过来（ETL 延迟 + 成本）。

Lakehouse 的做法：数据仍然以 Parquet 等 open format 存在 object storage 上，但通过 **table format（Iceberg / Delta Lake / Hudi）** 提供：
- ACID transactions
- Schema enforcement & evolution
- Time travel
- Partition pruning & predicate pushdown

这样 ML 框架（Spark MLlib、PyTorch）和 BI 工具（Trino、Presto）都可以**直接读同一份数据**，不需要 ETL 搬运。

---



## 2. Iceberg vs Delta Lake

 

Both are open table formats that bring ACID transactions to data lakes, but they differ in design philosophy. Iceberg uses a tree of metadata — a metadata file points to manifest lists, which point to manifest files, which track individual data files. This gives fine-grained file-level tracking and efficient partition pruning. Delta Lake, on the other hand, uses a transaction log — a `_delta_log` directory with sequential JSON files that record every operation. Iceberg is more engine-agnostic and supports hidden partitioning, while Delta Lake has tighter Spark integration and benefits from Databricks' ecosystem like Z-ordering and Liquid Clustering.

 

| 维度 | Apache Iceberg | Delta Lake |
|---|---|---|
| **Metadata 设计** | Tree 结构：metadata file → manifest list → manifest file → data files | Transaction log：`_delta_log/` 下的顺序 JSON 文件 |
| **Engine 兼容性** | Engine-agnostic（Spark, Flink, Trino, Presto, Dremio 原生支持） | 最初深度绑定 Spark；现在通过 UniForm 和 Delta Kernel 扩展兼容性 |
| **Partition 管理** | Hidden partitioning（用户写 query 不需要知道分区字段） | 显式分区，用户 query 需要感知分区列 |
| **Schema Evolution** | 完整支持（add / drop / rename / reorder columns），通过 column ID 追踪 | 支持 add / rename / change type，但依赖 column name 追踪 |
| **开源治理** | Apache 基金会项目，社区驱动 | Linux Foundation（2024年捐赠），此前由 Databricks 主导 |
| **Compaction** | 需要外部触发 rewrite（如 `rewrite_data_files`） | 有 `OPTIMIZE` 命令 + auto-compaction |
| **生态优势** | 跨引擎、hidden partition、partition evolution | Databricks 生态（Z-ordering、Liquid Clustering、Change Data Feed） |

**核心区别**：Iceberg 的 metadata tree 可以在不扫描所有 metadata 的情况下精确定位需要的文件（通过 manifest 里的 partition stats 做 pruning），而 Delta 的 transaction log 需要定期做 checkpoint（每 10 个 JSON commit 合并为一个 Parquet checkpoint）来加速读取。

---



## 3. Snapshot Isolation 怎么实现？

 

- Both Iceberg and Delta implement snapshot isolation through immutable snapshots. In Iceberg, every write creates a new snapshot with its own manifest list that points to the current set of data files. Readers always read from a specific snapshot, so they see a consistent view even while writers are appending new data. Writers use optimistic concurrency — they assume no conflict, do their work, then try to atomically swap the metadata pointer. If another writer committed first, the operation retries with conflict detection. Delta Lake works similarly but uses its transaction log — each commit appends a new JSON file to `_delta_log`, and readers pick a specific log version as their snapshot.

 

**Snapshot Isolation** 的核心：**读写互不阻塞，每个 reader 看到的是某个时间点的一致性快照。**

**Iceberg 实现方式：**
1. 每次 write 产生一个新 **snapshot**（不可变的）。
2. Snapshot 包含一个 manifest list，指向所有当前有效的 data files。
3. Reader 读某个 snapshot，看到的是那一刻的完整数据视图。
4. Writer 用 **optimistic concurrency control (OCC)**：
   - 基于当前 snapshot 做修改
   - 写完后尝试 **atomic swap**（更新 metadata file 里的 current-snapshot-id）
   - 如果发现别人已经提交了新 snapshot → **conflict detection** → 判断是否冲突 → 不冲突则 retry，冲突则失败
5. Atomic swap 在不同 catalog 上实现不同：Hive Metastore 用锁，AWS Glue / Nessie 用 compare-and-swap。
   <img src='./pic/1_3_iceberg_consistency.webp' width=500>

[Understanding Apache Iceberg’s Consistency Model](https://jack-vanlightly.com/analyses/2024/8/5/apache-icebergs-consistency-model-part-2) 

**Delta Lake 实现方式：**
1. 每次 commit 在 `_delta_log/` 目录下写一个新的 JSON 文件（如 `000000001.json`）。
2. Reader 读取到某个版本号为止的所有 log entry，重建当前 table state。
3. Writer 也用 OCC：尝试写下一个序号的 JSON 文件，如果文件已存在（别人先写了），则 retry。
4. 在 S3 上通过 `put-if-absent`（S3 conditional write）或 DynamoDB lock 实现原子性。
   <img src='./pic/1_3_deltalake_consistency.webp' width=500>

[Understanding Delta Lake's consistency model](https://jack-vanlightly.com/analyses/2024/4/29/understanding-delta-lakes-consistency-model)

**关键点**：两者都是 **copy-on-write** 语义 — 不修改已有文件，只追加新文件 + 更新 metadata 指针。

---



## 4. Schema Evolution 怎么做？

 

Iceberg handles schema evolution by assigning a unique column ID to every column. When you add, drop, rename, or reorder columns, Iceberg updates the schema in metadata but doesn't rewrite existing data files. Old data files still use the old schema, and Iceberg maps columns by ID, not by name or position. This means you can safely rename a column without breaking existing queries. Delta Lake also supports schema evolution — you can add columns, change data types with some constraints, and rename columns — but it historically tracked columns by name and position, which makes some operations like column reordering less seamless.

 

**Schema Evolution 的核心挑战**：表的 schema 会随业务变化（加字段、改类型、重命名），但已有的 Parquet 文件是按旧 schema 写的，不可能每次都 rewrite 所有历史数据。

**Iceberg 的做法（基于 column ID）：**
- 每个 column 分配一个 **唯一 integer ID**（在 Parquet 文件的 field_id 中存储）。
- Schema 变更只修改 metadata，不 rewrite data files。
- 读数据时，通过 column ID 映射，而不是 column name 或 position。
- 支持的操作：
  - **Add column**：新 ID，旧文件对应位置返回 null
  - **Drop column**：metadata 标记删除，旧文件中该列数据被忽略
  - **Rename column**：只改 metadata 中的 name，ID 不变，不影响旧文件
  - **Reorder columns**：改 metadata 中的顺序，ID 映射不变
  - **Type promotion**：如 int → long，float → double

**Delta Lake 的做法（基于 column name）：**
- Schema 存储在 transaction log 的 metadata action 中。
- 通过 `mergeSchema` 或 `overwriteSchema` 选项触发。
- 支持 add columns、rename（Spark 3.x+）、change type（有限制）。
- 因为按 name 追踪，rename 需要特殊处理（column mapping mode）。

**面试要点**：Iceberg 的 column ID 设计是其 schema evolution 的核心优势，使得 rename 和 reorder 天然安全。

---



## 5. Time Travel 原理

 

Time travel works because both formats keep a history of snapshots. In Iceberg, every commit creates an immutable snapshot. The metadata file maintains a snapshot log — a list of all historical snapshots with their timestamps. To query data as of a specific time or snapshot ID, the engine simply loads that snapshot's manifest list instead of the current one, and reads the data files it points to. Delta Lake achieves the same through its transaction log — you can query any version by replaying the log up to that version. The data files from old snapshots are not deleted until you explicitly run an expiration or vacuum process.

 

**Time Travel 的本质**：因为采用了 immutable files + snapshot 链的设计，历史版本天然被保留。

**Iceberg Time Travel 流程：**
1. 每次 commit → 新 snapshot（有唯一 snapshot ID + timestamp）。
2. Metadata file 维护 **snapshot log**（所有历史 snapshot 的列表）。
3. 查询历史数据：
   ```sql
   -- 按时间
   SELECT * FROM table TIMESTAMP AS OF '2024-01-01 00:00:00';
   -- 按 snapshot ID
   SELECT * FROM table VERSION AS OF 12345;
   ```
4. 引擎加载对应 snapshot 的 manifest list → 找到那一刻的 data files → 读取。
5. 旧 data files 不会被删除，除非你运行 **expire_snapshots**。

**Delta Lake Time Travel 流程：**
1. 每个 commit → `_delta_log/` 中的一个 JSON 文件。
2. 查询某个版本：replay log entries 从 0 到该版本。
3. 语法类似：
   ```sql
   SELECT * FROM table VERSION AS OF 5;
   SELECT * FROM table TIMESTAMP AS OF '2024-01-01';
   ```
4. 旧文件不删除，除非运行 `VACUUM`。

**Time Travel 的实际用途：**
- **Debug**：查看数据在某次写入前后的差异
- **Audit**：合规场景下追溯历史数据状态
- **Rollback**：发现错误写入后回退到正确版本
- **Reproducibility**：ML training 用固定 snapshot 保证可复现性

---



## 6. Time Travel, Backup, TTL

 

Time travel, backup, and TTL are related but serve different purposes. Time travel lets you query historical snapshots, but it's not a backup strategy — once you run snapshot expiration or vacuum, old data files get deleted and you lose that history. TTL, or time-to-live, controls how long snapshots and data files are retained. In Iceberg, you configure snapshot retention with `history.expire.max-snapshot-age-ms`, and expired snapshots' orphan files can be cleaned up. For actual backup, you need to replicate data to a separate storage location, because time travel only works as long as the underlying files exist.

 

这三个概念经常被混淆，面试中需要区分清楚：

**Time Travel（时间旅行）：**
- 查询历史 snapshot 的能力
- 依赖于旧 data files 仍然存在
- 不是 backup！只是"还没删"

**TTL / Retention（数据保留策略）：**
- Iceberg：`history.expire.max-snapshot-age-ms`（默认 5 天）控制保留多久的 snapshot
- Delta：`delta.logRetentionDuration`（默认 30 天）和 `delta.deletedFileRetentionDuration`（默认 7 天）
- 过期后，`expire_snapshots`（Iceberg）或 `VACUUM`（Delta）会清除不再被任何 snapshot 引用的文件
- 需要**同时**做两件事：(1) expire snapshots/log entries (2) 清理 orphan data files

**Backup（备份）：**
- 真正的 backup 需要将数据复制到**独立的存储位置**
- Time travel 不能替代 backup，因为：
  - Storage 损坏 → 所有 snapshot 都没了
  - 误执行 `VACUUM` / `expire_snapshots` → 历史数据被物理删除
- 常见策略：S3 cross-region replication、定期 snapshot export

**面试要点**：Time travel 是 retention window 内的便利功能，不是灾难恢复方案。

---



## 7. Metadata Tree（Iceberg）

 

Iceberg organizes metadata in a tree structure with three levels. At the top is the metadata file, which contains the table schema, partition spec, and a list of snapshots. Each snapshot points to a manifest list file, which is essentially a list of manifest files. Each manifest file tracks a set of data files and stores partition-level statistics like min/max values and row counts. This tree structure enables efficient query planning — the engine can prune entire manifest files based on partition stats without scanning actual data. The tree is immutable; every write creates new nodes rather than modifying existing ones.

 

Iceberg 的 metadata tree 是其核心设计，面试高频考点。

```text
            ┌─────────────────┐
            │  Metadata File  │  ← 最顶层，记录 schema、partition spec、snapshot 列表
            └────────┬────────┘
                     │
            ┌────────▼────────┐
            │  Snapshot       │  ← 每次 commit 产生，指向一个 manifest list
            └────────┬────────┘
                     │
            ┌────────▼────────┐
            │  Manifest List  │  ← Avro 文件，列出所有 manifest files
            └────────┬────────┘
                     │
         ┌───────────┼───────────┐
         ▼           ▼           ▼
   ┌──────────┐┌──────────┐┌──────────┐
   │ Manifest ││ Manifest ││ Manifest │  ← 每个 manifest 追踪一批 data files
   │  File    ││  File    ││  File    │     包含 partition stats（min/max/null count）
   └────┬─────┘└────┬─────┘└────┬─────┘
        │           │           │
    data files  data files  data files    ← 实际的 Parquet/ORC/Avro 文件
```

**各层的职责：**

| 层级 | 格式 | 内容 |
|---|---|---|
| **Metadata File** | JSON（v1）或 Avro | Table schema、partition spec、sort order、current snapshot ID、snapshot history |
| **Manifest List** | Avro | 列出所有 manifest files + 每个 manifest 的 partition range summary |
| **Manifest File** | Avro | 每个 data file 的路径、大小、record count、partition values、column-level stats（min/max/null count） |
| **Data Files** | Parquet / ORC / Avro | 实际数据 |

**为什么这个设计高效？**
1. **Manifest-level pruning**：query planner 先看 manifest list 中每个 manifest 的 partition summary → 跳过不相关的 manifest。
2. **File-level pruning**：在需要的 manifest 中，用 column stats（min/max）进一步跳过不需要的 data files。
3. **Immutability**：所有层都是不可变的，新 commit 创建新节点，旧的保留（支持 time travel）。

---



## 8. Manifest File（详解）

 

A manifest file in Iceberg is an Avro file that tracks a subset of data files belonging to a table. Each entry in a manifest records the data file's path, format, partition values, record count, file size, and column-level statistics including lower and upper bounds and null counts. These stats are crucial for scan planning — the engine uses them to skip files that can't contain matching rows, a process called file pruning. Manifest files are also immutable; when data files are added or deleted, new manifests are written rather than modifying existing ones. A manifest also records whether each file entry is an "added" or "deleted" status to support snapshot diffs.

 

Manifest file 是 Iceberg metadata tree 中最关键的一层，直接决定查询性能。

**Manifest File 中每条 entry 包含：**

| 字段 | 说明 |
|---|---|
| `file_path` | Data file 在 object storage 上的路径 |
| `file_format` | Parquet / ORC / Avro |
| `partition` | 该文件对应的 partition values（如 `date=2024-01-15`） |
| `record_count` | 该文件中的行数 |
| `file_size_in_bytes` | 文件大小 |
| `column_sizes` | 每列的字节大小 |
| `value_counts` | 每列的非 null 值数量 |
| `null_value_counts` | 每列的 null 值数量 |
| `lower_bounds` | 每列的最小值 |
| `upper_bounds` | 每列的最大值 |
| `status` | 0=EXISTING, 1=ADDED, 2=DELETED |
| `snapshot_id` | 关联的 snapshot ID |

**Query Planning 示例：**
```sql
SELECT * FROM orders WHERE order_date = '2024-06-15' AND amount > 1000;
```
1. 读 manifest list → 根据 partition summary 跳过不包含 `2024-06-15` 的 manifest files。
2. 读剩余 manifest files → 根据 `lower_bounds` / `upper_bounds` 跳过 `amount` 最大值 ≤ 1000 的 data files。
3. 只读最终剩下的 data files → 再用 Parquet 内部的 row group stats 进一步过滤。

**Status 字段的作用：**
- `ADDED`：该文件是在某个 snapshot 中新增的
- `DELETED`：该文件被某个 snapshot 标记删除（实际文件可能还在，等 expire）
- `EXISTING`：从上一个 snapshot 继承的，没有变化
- 这使得 **incremental scan** 成为可能（只看 ADDED / DELETED 的文件）

---



## 9. Partition Spec

 

Partition spec in Iceberg defines how data is partitioned, but unlike traditional Hive-style partitioning, Iceberg uses hidden partitioning. You define partition transforms — like year, month, day, hour on a timestamp column, or bucket and truncate on other columns — and Iceberg automatically applies them during writes. Users don't need to include partition columns in their queries; the engine automatically translates filter predicates to partition pruning. A major advantage is partition evolution — you can change the partition spec without rewriting existing data. New data uses the new spec while old data retains the old spec, and Iceberg handles both transparently during reads.

 

**传统 Hive Partitioning 的问题：**
```sql
-- Hive 风格：用户必须知道分区结构
SELECT * FROM events WHERE year=2024 AND month=6 AND day=15;
-- 如果写成这样，分区不会被利用：
SELECT * FROM events WHERE event_time = '2024-06-15 10:00:00';
```
用户必须显式引用分区列，否则 partition pruning 失效，导致全表扫描。

**Iceberg Hidden Partitioning：**
```sql
-- 定义 partition spec（DDL 层面）
ALTER TABLE events ADD PARTITION FIELD hour(event_time);

-- 用户 query 不需要知道分区结构
SELECT * FROM events WHERE event_time > '2024-06-15 10:00:00';
-- Iceberg 自动推断：hour(event_time) 对应哪些分区 → pruning
```

**Partition Transforms（分区转换函数）：**

| Transform | 说明 | 示例 |
|---|---|---|
| `year(ts)` | 提取年份 | `2024` |
| `month(ts)` | 年-月 | `2024-06` |
| `day(ts)` | 年-月-日 | `2024-06-15` |
| `hour(ts)` | 年-月-日-时 | `2024-06-15-10` |
| `bucket(N, col)` | Hash 分桶 | `bucket(16, user_id)` → 0-15 |
| `truncate(W, col)` | 截断 | `truncate(10, zip_code)` → "9021" |
| `identity(col)` | 原值分区 | 等同于 Hive 风格 |

**Partition Evolution（分区演化）：**

这是 Iceberg 的重要特性。假设业务增长，需要从 `month` 分区改为 `day` 分区：

1. 执行 `ALTER TABLE events ADD PARTITION FIELD day(event_time)`
2. **旧数据不 rewrite**，仍然按 `month` 分区。
3. 新写入的数据按 `day` 分区。
4. 查询时 Iceberg 自动处理两种分区 spec，分别做对应的 pruning。

**Delta Lake 的对比：**
- Delta 使用显式的 `PARTITIONED BY (col)` 语法。
- 不支持 hidden partitioning 和 transform。
- 分区变更需要 rewrite 整个表（或使用 Liquid Clustering 作为替代方案）。

---



## 10. Compaction 为什么需要？

 

Compaction is necessary because streaming ingestion and frequent small writes create many small files, known as the small file problem. Each small file adds overhead — more metadata to track, more file opens during reads, and less efficient compression. Compaction merges these small files into larger, optimally-sized files, typically 256 MB to 1 GB. In Iceberg, you run `rewrite_data_files` which creates new data files and a new snapshot pointing to them. In Delta Lake, the `OPTIMIZE` command does the same thing and can also apply Z-ordering to co-locate related data for better pruning. Without regular compaction, query performance degrades significantly over time.

 

**Small File Problem（小文件问题）：**

在实际生产中，数据通常以 streaming 或 micro-batch 方式写入（如 Kafka → Spark Structured Streaming → Iceberg table），每次 commit 可能只写几 MB 甚至几 KB 的文件。随着时间推移：
- 文件数量爆炸（可能几十万个小文件）
- 每个文件需要一条 manifest entry → metadata 膨胀
- 读取时需要打开大量文件 → I/O overhead
- Parquet 文件太小 → column encoding 和 compression 效率低
- Object storage（S3）有 per-request 成本 → 小文件 = 更多请求 = 更贵

**Compaction 做什么：**
1. 读取一批小文件
2. 合并为少量大文件（target size 通常 256MB - 1GB）
3. 写新的 data files
4. 创建新 snapshot（指向合并后的文件）
5. 旧小文件在 snapshot expire 后被清除

**Iceberg Compaction：**
```sql
CALL catalog.system.rewrite_data_files(
  table => 'db.events',
  options => map('target-file-size-bytes', '536870912')  -- 512MB
);
```
- 可配置 `partial-progress.enabled` 支持大表分批 compaction
- 可结合 sort order 做 **sorted compaction**（提升后续查询的 pruning 效率）

**Delta Lake Compaction：**
```sql
OPTIMIZE db.events;
-- 带 Z-ordering
OPTIMIZE db.events ZORDER BY (user_id, event_type);
```
- `OPTIMIZE` 合并小文件
- `ZORDER` 额外按指定列做多维排序（co-locate 相关数据）
- Databricks 还有 **auto-compaction** 和 **Liquid Clustering**

**Compaction 策略：**
- **定时任务**：每小时/每天跑一次 compaction job
- **阈值触发**：当小文件数超过某个阈值时触发
- **Partition 级别**：只 compact 最近更新的 partitions，避免全表 rewrite
- 注意：compaction 期间 table 仍可读（snapshot isolation 保证）

---



## 11. 补充高频知识点

### Copy-on-Write vs Merge-on-Read

Copy-on-write rewrites entire data files on every update or delete, giving fast reads but slow writes. Merge-on-read writes small delta files and merges them at read time, giving fast writes but slower reads.   

Iceberg supports both modes; Delta Lake primarily uses copy-on-write with deletion vectors as a merge-on-read optimization.


- **Copy-on-Write (COW)**：UPDATE/DELETE 时 rewrite 整个受影响的 data file → 读快，写慢
- **Merge-on-Read (MOR)**：写入时只写 delta file（或 delete file），读取时合并 → 写快，读慢
- Iceberg v2 引入了 **positional delete files** 和 **equality delete files** 来支持 MOR
- Delta Lake 引入了 **deletion vectors**（标记哪些行被删除，避免 rewrite 整个文件）

### Catalog 的作用


A catalog maps table names to metadata file locations. It's the entry point for discovering and accessing tables. Common catalogs include Hive Metastore, AWS Glue, Nessie, and REST catalog.


- Catalog 是 table name → metadata file location 的映射
- 也负责 atomic commit（通过 compare-and-swap 更新 metadata pointer）
- 选型：Hive Metastore（传统）、AWS Glue（AWS 生态）、Nessie（Git-like branching）、REST Catalog（Iceberg 标准化趋势）

### Z-Ordering / Liquid Clustering


- **Z-ordering**：将多列的值映射到 Z 曲线上排序，使得在这些列上过滤时 data skipping 效率更高。是 Delta Lake / Databricks 的特色功能。
- **Liquid Clustering**：Databricks 新推出的自动聚类方案，取代 Z-ordering + 手动 partition，自动决定如何组织数据。
- Iceberg 的对应方案：**sorted compaction** + **partition transforms**。

---



## 12. 快速回顾 Cheat Sheet

| 概念 | 一句话总结 |
|---|---|
| **Lakehouse** | Data lake 存储 + warehouse 级别事务性（通过 table format 实现） |
| **Iceberg vs Delta** | Iceberg = metadata tree + hidden partitioning；Delta = transaction log + Databricks 生态 |
| **Snapshot Isolation** | 每次 commit 产生 immutable snapshot，OCC 保证并发安全 |
| **Schema Evolution** | Iceberg 用 column ID 追踪（rename/reorder 安全）；Delta 用 column name |
| **Time Travel** | 查询历史 snapshot，依赖旧文件未被清除 |
| **TTL / Retention** | 控制 snapshot 保留时长，过期后可物理清除旧文件 |
| **Metadata Tree** | Metadata file → manifest list → manifest files → data files（三层 pruning） |
| **Manifest File** | 追踪 data files 的 Avro 文件，包含 partition values 和 column stats |
| **Partition Spec** | Iceberg 的 hidden partitioning + transform 函数，支持 partition evolution |
| **Compaction** | 合并小文件为大文件，解决 small file problem |
| **COW vs MOR** | COW 读快写慢，MOR 写快读慢 |
| **Catalog** | Table name → metadata location 的映射，负责 atomic pointer swap |

